### 1) Tokenizer 내려받고 정보 조회

`AutoTokenizer.from_pretrained()`를 통해서 사전에 학습된 토크나이저는 설정, 어휘, 병합표(merges) 등을 Hub에서 파일을 받아옴. 

In [2]:
import torch
from transformers import AutoTokenizer

In [3]:
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-4b-it")

In [4]:
# 내려받은걸 다음과 같이 확인도 가능
!ls /purestorage/AILAB/AI_1/tyk/0_Software/cache/huggingface/hub/models--google--gemma-3-4b-it/snapshots/093f9f388b31de276ce2de164bdc2081324b9767

# https://huggingface.co/ + google/gemma-3-4b-it 여기 가서도 확인 가능

added_tokens.json		  preprocessor_config.json
chat_template.json		  processor_config.json
config.json			  special_tokens_map.json
generation_config.json		  tokenizer.json
model-00001-of-00002.safetensors  tokenizer.model
model-00002-of-00002.safetensors  tokenizer_config.json
model.safetensors.index.json


In [5]:
# 가끔 이름으로 주소를 추측하기 힘든 경우
from huggingface_hub import model_info, hf_hub_url

print(model_info("bert-base-cased").modelId)

google-bert/bert-base-cased


`tokenizer_config.json` : 토크나이저 클래스, 특수 토큰 이름이 들어있음. "model_max_length": 1024, "padding_side": "right", "tokenizer_class": "GPT2Tokenizer"

`special_tokens_map.json` : `<unk>`, `<pad>`, `<eos>`같은 특수 토큰의 실제 vocab 상의 문자열과 해당 토큰의 목적을 매핑.

`added_tokens.json` : 기본 vocab 이후 사용자가 tokenizer.add_tokens() 추가한 토큰 목록만 따로 저장.


`tokenizer.json` : fast 토크나이저 전용 JSON, vocab, merege table, 특수토큰, normalizer 설정이 모두 있음

`tokenizer.model` : 바이너리 파일, (모델 파라미터 + vocab 포함), SentencePiece 토크나이저에 해당.

`vocab.json` : BPE(Byte Pair Encoding)계열의 토크나이저(예: GPT-2, RoBERTa)에서 사용되는 JSON 파일. vocab.json(토큰=>ID)

`vocab.txt` : WordPiece 계열의 토크나이저(예: BERT, DistilBERT)에서 주로 사용되는 텍스트 파일. 각 줄이 하나의 토큰

`merges.txt` : BPE(Byte Pair Encoding) 계열의 토크나이저에서 병합되는 규칙. SentencePiece에는 해당 안됨


모델 마다 저게 다 있는건 아님. BERT는 `added_tokens.json`가 없음. 

In [6]:
# tokenizer.json에 살펴보면 어떤 vocab들이 있는지 확인
tokenizer.vocab_size

262144

In [7]:
# 직접 조회해서 확인도 가능
from transformers import BertTokenizer
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-cased")
bert_tokenizer

BertTokenizer(name_or_path='bert-base-cased', vocab_size=28996, model_max_length=512, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [24]:
tokenizer.special_tokens_map # eoi_token은 모델이 내부적으로 쓰기 때문에 직접 건들 필요가 없다

{'bos_token': '<bos>',
 'eos_token': '<eos>',
 'unk_token': '<unk>',
 'pad_token': '<pad>',
 'boi_token': '<start_of_image>',
 'eoi_token': '<end_of_image>',
 'image_token': '<image_soft_token>'}

In [ ]:
# tokenizer.json이 있으면 여기로 로드 가능
tokenizer_directory_path = "/purestorage/AILAB/AI_1/tyk/0_Software/cache/huggingface/hub/models--google--gemma-3-4b-it/snapshots/093f9f388b31de276ce2de164bdc2081324b9767/"

local_tokenizer = AutoTokenizer.from_pretrained(tokenizer_directory_path)

In [11]:
local_tokenizer.vocab_size

262144

https://github.dev/huggingface/transformers에 가서 해당 클래스에 대한 정의를 확인. `huggingface/transformers/src/transformers/models/bert` 위치에 BERT 모델들이 모두 정의되어 있음. `modeling_bert.py` : BERT Transformer 모델. `tokenization_bert.py` : 해당 모델이 사용한 토크나이저

### 2) Tokenizer 사용하기

실제 동작하면 Tokenization + Conversion 까지 모두 수행 : Encoding

![Image](https://img1.daumcdn.net/thumb/R1280x0/?scode=mtistory2&fname=https%3A%2F%2Fblog.kakaocdn.net%2Fdn%2FbPonBM%2FbtrGw3n6X5G%2FTmWKTcdq48tW2lhlCFFhQK%2Fimg.png)

[내부적](https://github.com/huggingface/transformers/blob/e8e0c76162263840661fc0ca0da3952861754759/src/transformers/models/bert/tokenization_bert.py#L31)으로 토크나이저 클래스의 생성자에서 vocab.txt에서 불러와서 vocab idx<-> token dict를 들고 있음. 이걸로 매핑해서 정수 값으로 치환함

In [12]:
tokenizer.encode('Derinkuyu is an underground city.')

[2, 17361, 961, 78658, 563, 614, 26407, 3207, 236761]

In [13]:
tokenizer.encode(' hello'), tokenizer.encode('hello')

([2, 29104], [2, 23391])

In [14]:
tokenizer.decode(2)

'<bos>'

In [15]:
# a trailing space을 넣지 말자
tokenizer.encode('The capital of France is '), tokenizer.decode(236743)

([2, 818, 5279, 529, 7001, 563, 236743], ' ')

In [ ]:
# gemma tokenizer input_ids, attention_mask 반환
tokenizer("Using a Transformer network is simple")

{'input_ids': [2, 17123, 496, 92474, 3707, 563, 3606], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}

In [ ]:
# gemma tokenizer input_ids, attention_mask, token_type_ids 반환
bert_tokenizer("Using a Transformer network is simple")

{'input_ids': [101, 7993, 170, 13809, 23763, 2443, 1110, 3014, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
# 정말 진짜 토크나이징
sequence = "Using a Transformer network is simple"
tokens = tokenizer.tokenize(sequence)

print(tokens)

['Using', '▁a', '▁Transformer', '▁network', '▁is', '▁simple']


In [19]:
sequence = "Using a Transformer network is simple"
tokens = bert_tokenizer.tokenize(sequence)

print(tokens)

['Using', 'a', 'Trans', '##former', 'network', 'is', 'simple']


In [20]:
# 새로운 토큰 추가
# 이는 기존의 임베딩 테이블의 vocab을 훼손하지 않고 그대로 두면서 몇개를 커스텀으로 더 추가하기 위함.
# 파인튜닝에 새로운 토큰이 필요하다고 하면 사용하게 됨.

print(f"초기 토크나이저의 어휘(vocab) 크기: {len(tokenizer.vocab)}")

new_tokens = ["<MY_SPECIAL_TOKEN>", "[NEW_CATEGORY]", "인공지능"]
num_added_tokens = tokenizer.add_tokens(new_tokens)

print(f"추가된 토큰 개수: {num_added_tokens}")
print(f"새 토크나이저의 어휘(vocab) 크기: {len(tokenizer.vocab)}")



초기 토크나이저의 어휘(vocab) 크기: 262145
추가된 토큰 개수: 3
새 토크나이저의 어휘(vocab) 크기: 262148


In [21]:
for token in new_tokens:
    token_id = tokenizer.convert_tokens_to_ids(token)
    print(f"'{token}'의 ID: {token_id}")
    # 토크나이저의 decode도 테스트해봅니다.
    print(f"ID {token_id} 디코딩: '{tokenizer.decode([token_id])}'")

'<MY_SPECIAL_TOKEN>'의 ID: 262145
ID 262145 디코딩: '<MY_SPECIAL_TOKEN>'
'[NEW_CATEGORY]'의 ID: 262146
ID 262146 디코딩: '[NEW_CATEGORY]'
'인공지능'의 ID: 262147
ID 262147 디코딩: '인공지능'


In [22]:
SAVE_DIR = "/purestorage/AILAB/AI_1/tyk/3_CUProjects/language_model/LLM/3_datasets/tokenizer/out"
tokenizer.save_pretrained(SAVE_DIR)

('/purestorage/AILAB/AI_1/tyk/3_CUProjects/language_model/LLM/3_datasets/tokenizer/out/tokenizer_config.json',
 '/purestorage/AILAB/AI_1/tyk/3_CUProjects/language_model/LLM/3_datasets/tokenizer/out/special_tokens_map.json',
 '/purestorage/AILAB/AI_1/tyk/3_CUProjects/language_model/LLM/3_datasets/tokenizer/out/chat_template.jinja',
 '/purestorage/AILAB/AI_1/tyk/3_CUProjects/language_model/LLM/3_datasets/tokenizer/out/tokenizer.json')

In [23]:
loaded_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
print(f"로드된 토크나이저의 어휘(vocab) 크기: {len(loaded_tokenizer.vocab)}")

로드된 토크나이저의 어휘(vocab) 크기: 262148


In [24]:
# 실제 쓰려면 모델의 임베딩도 맞춰서 같이 재조정해야함.
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")

print(f"변경 전 모델의 임베딩 크기: {model.get_input_embeddings().weight.shape[0]}")
print(f"변경 전 토크나이저의 어휘 크기: {len(tokenizer)}")

# 새로운 토큰 추가
new_tokens = ["<special_word>", "<new_concept>"]
num_added_tokens = tokenizer.add_tokens(new_tokens)
print(f"추가된 토큰 개수: {num_added_tokens}")
print(f"변경 후 토크나이저의 어휘 크기: {len(tokenizer)}")

# 모델의 임베딩 레이어 크기 조절
# 새로 추가된 토큰에 대해 무작위로 초기화된 임베딩 벡터가 생성됩니다.
model.resize_token_embeddings(len(tokenizer))
print(f"변경 후 모델의 임베딩 크기: {model.get_input_embeddings().weight.shape[0]}")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

변경 전 모델의 임베딩 크기: 50257
변경 전 토크나이저의 어휘 크기: 50257
추가된 토큰 개수: 2
변경 후 토크나이저의 어휘 크기: 50259


The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


변경 후 모델의 임베딩 크기: 50259


### 3) FastTokenizer

- `AutoTokenizer.from_pretrained()`에서 `use_fast=False` 사용
- Rust 언어로 구현되어서 더 빠름
- 병렬 학습시에는 환경변수 조절하기

```python
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
```

In [25]:
import time
from transformers import GPT2Tokenizer, GPT2TokenizerFast

# 1. 비교할 텍스트 준비
# 충분히 긴 텍스트를 사용해야 속도 차이를 명확하게 확인할 수 있습니다.
long_text = """
The quick brown fox jumps over the lazy dog.
This is a longer piece of text to demonstrate the speed difference between
the regular Python-based tokenizer and the Rust-based fast tokenizer.
Modern large language models rely heavily on efficient tokenization,
and the speed of this process can significantly impact training and inference times.
Hugging Face's `tokenizers` library, which powers the fast tokenizers,
is written in Rust for optimal performance.
Let's see how much faster it really is!
""" * 100 # 텍스트를 100번 반복하여 길이를 늘림

num_runs = 100 # 각 토크나이저를 실행할 횟수

print(f"총 텍스트 길이: {len(long_text)} 문자")
print(f"각 토크나이저 실행 횟수: {num_runs} 회\n")

# 2. Python 기반 Tokenizer (GPT2Tokenizer) 로드 및 테스트
print("--- Python 기반 GPT2Tokenizer 테스트 ---")
python_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
python_times = []

for _ in range(num_runs):
    start_time = time.time()
    # tokenize()는 토큰 문자열 리스트를 반환
    # encode() 또는 __call__()은 input_ids를 포함한 dict를 반환
    _ = python_tokenizer.encode(long_text)
    end_time = time.time()
    python_times.append(end_time - start_time)

avg_python_time = sum(python_times) / num_runs
print(f"Python Tokenizer 평균 실행 시간: {avg_python_time:.6f} 초")
print(f"토큰 개수 (예시): {len(python_tokenizer.encode(long_text))}\n")


# 3. Rust 기반 Fast Tokenizer (GPT2TokenizerFast) 로드 및 테스트
print("--- Rust 기반 GPT2TokenizerFast 테스트 ---")
fast_tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
fast_times = []

for _ in range(num_runs):
    start_time = time.time()
    # Fast Tokenizer도 동일하게 encode() 또는 __call__() 사용
    _ = fast_tokenizer.encode(long_text)
    end_time = time.time()
    fast_times.append(end_time - start_time)

avg_fast_time = sum(fast_times) / num_runs
print(f"Fast Tokenizer 평균 실행 시간: {avg_fast_time:.6f} 초")
print(f"토큰 개수 (예시): {len(fast_tokenizer.encode(long_text))}\n")


# 4. 결과 비교
if avg_python_time > 0:
    speed_up_factor = avg_python_time / avg_fast_time
    print(f"Fast Tokenizer가 Python Tokenizer보다 약 {speed_up_factor:.2f} 배 빠릅니다.")
else:
    print("Python Tokenizer 시간이 너무 작아 배율 계산 불가.")

총 텍스트 길이: 50000 문자
각 토크나이저 실행 횟수: 100 회

--- Python 기반 GPT2Tokenizer 테스트 ---


Token indices sequence length is longer than the specified maximum sequence length for this model (10800 > 1024). Running this sequence through the model will result in indexing errors


Python Tokenizer 평균 실행 시간: 0.050682 초
토큰 개수 (예시): 10800

--- Rust 기반 GPT2TokenizerFast 테스트 ---


Token indices sequence length is longer than the specified maximum sequence length for this model (10800 > 1024). Running this sequence through the model will result in indexing errors


Fast Tokenizer 평균 실행 시간: 0.024616 초
토큰 개수 (예시): 10800

Fast Tokenizer가 Python Tokenizer보다 약 2.06 배 빠릅니다.


### Tokenizer 학습시키기

In [26]:
# 학습

import pandas as pd
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/e9t/nsmc/master/ratings.txt", filename="ratings.txt")

naver_df = pd.read_table('ratings.txt')
naver_df = naver_df.dropna(how='any')
with open('naver_review.txt', 'w', encoding='utf8') as f:
    f.write('\n'.join(naver_df['document']))


In [27]:
data_file = 'naver_review.txt'
vocab_size = 30000
limit_alphabet = 6000
min_frequency = 5

tokenizer.train(files=data_file,
                vocab_size=vocab_size,
                limit_alphabet=limit_alphabet, # 병합 전의 초기 토큰의 허용 개수
                min_frequency=min_frequency) # 최소 해당 횟수만큼 등장한 쌍(pair)의 경우에만 병합 대상이 된다.

AttributeError: GPT2TokenizerFast has no attribute train

In [ ]:
# 학습 완료 후 vocab 저장
tokenizer.save_model('/purestorage/AILAB/AI_1/tyk/3_CUProjects/language_model/LLM/3_datasets/tokenizer/out2')

In [ ]:
# vocab 로드
df = pd.read_fwf('vocab.txt', header=None)
df

In [ ]:
encoded = tokenizer.encode('아 배고픈데 짜장면먹고싶다')
print('토큰화 결과 :',encoded.tokens)
print('정수 인코딩 :',encoded.ids)
print('디코딩 :',tokenizer.decode(encoded.ids))

In [ ]:
encoded = tokenizer.encode('커피 한잔의 여유를 즐기다')
print('토큰화 결과 :',encoded.tokens)
print('정수 인코딩 :',encoded.ids)
print('디코딩 :',tokenizer.decode(encoded.ids))

### 다른 Tokenizer 사용

In [ ]:
# llama 시리즈에서 쓰는 spm
import sentencepiece as spm

spm_tokenizer = spm.SentencePieceProcessor(model_file="tokenizer.model")

In [ ]:
# GPT에서 쓰는 BPE (허깅페이스 제공 버전)
from transformers import GPT2Tokenizer, GPT2TokenizerFast

# 부족한 부분: strings 변수 정의
strings = "Hello, this is a test sentence for tokenization."
# 또는 strings = ["First sentence.", "Second sentence."] 와 같이 리스트로 정의할 수도 있습니다.

hf_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
print("GPT2Tokenizer (Python):", hf_tokenizer(strings)["input_ids"])

hf_tokenizer_fast = GPT2TokenizerFast.from_pretrained("gpt2")
print("GPT2TokenizerFast (Rust/C++):", hf_tokenizer_fast(strings)["input_ids"])

In [ ]:
# BPE Tokenizer 테스트
import tiktoken

tik_tokenizer = tiktoken.get_encoding("gpt2")
text = "Hello, world. Is this-- a test?"

integers = tik_tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print("Encoded integers:", integers)


strings = tik_tokenizer.decode(integers)

print("Decoded strings:", strings)

print(tik_tokenizer.n_vocab)

### 4) Processor

VLM에선 Tokenizer와 Image Processor를 둘 다 가지고 있음. 

In [7]:
from transformers import AutoProcessor
processor = AutoProcessor.from_pretrained("google/gemma-3-4b-it")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [8]:
type(processor)

transformers.models.gemma3.processing_gemma3.Gemma3Processor

In [ ]:
processor.tokenizer.encode('아 배고픈데 짜장면먹고싶다')